# Phần B — Region Growing và SRG

### Bài tập 2 — Cài đặt Region Growing và Seeded Region Growing

- **Created**: 2026-11-11T07:35:41+07:00
- **Last Updated**: 2026-11-11T08:03:09+07:00
- **Author**: Xem [nhóm và phân công](../README.md#nhom-va-phan-cong).
- **Objective**: Tự cài đặt, kiểm chứng và phân tích phần B theo đề bài.

## References & Consumed Artifacts

- [Đề bài gốc](../pdf/Computer_Vision%2045.pdf).
- [Hướng dẫn và hợp đồng hàm](../docs/shared/HUONG_DAN_THUC_HIEN.md).
- [Quy ước notebook](../agents/rules/NOTEBOOK_HEADER_CONVENTION.md).
- [Quy tắc báo cáo](../agents/rules/RESULTS_REPORTING.md).
- [Ảnh phần này](../images/phan_b/), [kết quả xuất](../experiments/results/phan_b/).
- Source Code: bốn hàm phần B và các hàm hỗ trợ được định nghĩa trực tiếp trong notebook; chưa có kết quả thí nghiệm.

Hai notebook là bản làm việc; trước khi nộp phải ghép thành một notebook theo đề.


## 1. Thư viện và quy ước cài đặt

Các hàm được cài đặt trực tiếp trong notebook. Không gọi hàm góc /
phân vùng dựng sẵn bên trong hàm phải nộp. Đối chiếu thư viện chỉ ở phần được đề cho phép.

Được dùng NumPy, Matplotlib, math, time; heapq cho SRG. Các hàm SciPy/OpenCV có điều kiện
phải đối chiếu bảng thư viện ở README và đề. Cài thư viện không đồng nghĩa được dùng mọi hàm.

Nhóm tự chọn ảnh, giả thuyết, phép đo, tham số và kết luận.

## 2. Môi trường và dữ liệu — nhóm tự điền

Metadata kernel là Python 3; phiên bản trong metadata phản ánh môi trường lúc dựng khung,
không thay thế việc ghi phiên bản thực tế khi chạy bài.

| Thành phần | Phiên bản / thông tin thực tế |
| --- | --- |
| Python / đường dẫn kernel | Chưa ghi |
| NumPy / SciPy / Matplotlib / OpenCV | Chưa ghi |
| Thiết bị, thời điểm chạy, seed ngẫu nhiên | Chưa ghi |

| Ảnh | Nguồn / quyền sử dụng | Kích thước / kiểu / miền cường độ | Phép chỉnh sửa |
| --- | --- | --- | --- |
| Chưa chọn | Chưa điền | Chưa điền | Chưa điền |

- Mở notebook với working directory là thư mục chứa notebook.
- Đọc ảnh từ `../images/phan_b/`; xuất bảng/hình vào `../experiments/results/phan_b/`.
- Ghi rõ ảnh xám `(H, W)`, miền cường độ và ngưỡng cùng đơn vị.
- Điểm theo `(row, col)`; khi vẽ trục ngang là col, trục dọc là row.
- Quy ước biên, seeds, biến đổi và ghép/đếm: nhóm tự ghi trước khi đo.

Ô dưới khai báo NumPy, heapq, deque và bộ máy SRG dùng chung.

Ảnh float64 giữ nguyên miền cường độ. Seeds là các tọa độ nguyên (row, col), không rỗng, không trùng, nằm trong ảnh; nhãn theo thứ tự seeds. Lân cận 4: trên, trái, phải, dưới; lân cận 8 bổ sung chéo trên trái, trên phải, dưới trái, dưới phải. SRG dùng heap lazy; biên là pixel kề nhiều vùng tại thời điểm gán và vẫn có nhãn. Phá hòa insertion theo thứ tự chèn/seed, raster theo tọa độ pixel/seed, random theo ưu tiên cố định trong mỗi lần chạy. Đặt np.random.seed trước khi chạy để tái lập. Các quy ước frames và sensitivity được ghi trong docstring.


In [1]:
import heapq
from collections import deque
import numpy as np


def _b_integer(value, name, minimum=0):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < minimum:
        raise ValueError(f"{name} phải là số nguyên >= {minimum}")
    return int(value)


def _b_inputs(img, seeds):
    image = np.asarray(img, dtype=np.float64)
    if image.ndim != 2 or image.size == 0 or not np.isfinite(image).all():
        raise ValueError("img phải là ảnh xám 2D hữu hạn, không rỗng")
    points = np.asarray(seeds)
    if (points.ndim != 2 or points.shape[1] != 2 or len(points) == 0
            or not np.issubdtype(points.dtype, np.integer)
            or np.any(points < 0) or np.any(points >= image.shape)):
        raise ValueError("seeds phải là mảng tọa độ nguyên (N, 2) nằm trong ảnh")
    if len(np.unique(points, axis=0)) != len(points):
        raise ValueError("seeds không được trùng vị trí")
    return image, points.astype(np.intp, copy=True)


def _b_neighbors(row, col, shape, connectivity=4):
    offsets = [(-1, 0), (0, -1), (0, 1), (1, 0)]
    if connectivity == 8:
        offsets += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    for dr, dc in offsets:
        r, c = row + dr, col + dc
        if 0 <= r < shape[0] and 0 <= c < shape[1]:
            yield r, c


def _b_srg_engine(img, seeds, connectivity=4, stale="lazy", tie="insertion", on_step=None):
    image, seeds = _b_inputs(img, seeds)
    if connectivity not in (4, 8):
        raise ValueError("connectivity phải là 4 hoặc 8")
    if stale not in ("lazy", "frozen"):
        raise ValueError("stale phải là lazy hoặc frozen")
    if tie not in ("insertion", "raster", "random"):
        raise ValueError("tie phải là insertion, raster hoặc random")
    labels = np.full(image.shape, -1, dtype=np.int32)
    boundary = np.zeros(image.shape, dtype=bool)
    for region, (row, col) in enumerate(seeds):
        labels[row, col] = region
    totals = image[seeds[:, 0], seeds[:, 1]].copy()
    seed_values = totals.copy()
    counts = np.ones(len(seeds), dtype=np.int64)
    versions = np.zeros(image.shape, dtype=np.int64)
    insertion = np.full(image.shape, -1, dtype=np.int64)
    serial = 0
    heap = []
    if tie == "random":
        pixel_priority = np.random.random(image.shape)
        region_priority = np.random.random(len(seeds))

    def best_region(row, col):
        adjacent = sorted({int(labels[r, c]) for r, c in
                           _b_neighbors(row, col, image.shape, connectivity) if labels[r, c] >= 0})
        if not adjacent:
            raise RuntimeError("Pixel frontier không kề vùng nào")
        def key(region):
            reference = totals[region] / counts[region] if stale == "lazy" else seed_values[region]
            delta = abs(float(image[row, col] - reference))
            if tie == "raster":
                priority = tuple(seeds[region])
            elif tie == "random":
                priority = (float(region_priority[region]),)
            else:
                priority = (region,)
            return (delta, *priority, region)
        winner = min(adjacent, key=key)
        return key(winner)[0], winner, len(adjacent) > 1

    def enqueue(row, col):
        nonlocal serial
        if labels[row, col] >= 0:
            return
        if insertion[row, col] < 0:
            insertion[row, col] = serial
            serial += 1
        delta, region, _ = best_region(row, col)
        versions[row, col] += 1
        if tie == "raster":
            priority = (row, col)
        elif tie == "random":
            priority = (float(pixel_priority[row, col]), int(insertion[row, col]))
        else:
            priority = (int(insertion[row, col]), 0)
        heapq.heappush(heap, (delta, *priority, row, col, int(versions[row, col]), region))

    for row, col in seeds:
        for r, c in _b_neighbors(row, col, image.shape, connectivity):
            enqueue(r, c)
    assigned = len(seeds)
    if on_step is not None:
        on_step(labels, assigned)
    while heap:
        saved_delta, _, _, row, col, version, saved_region = heapq.heappop(heap)
        if labels[row, col] >= 0 or version != versions[row, col]:
            continue
        delta, region, is_boundary = best_region(row, col)
        if delta != saved_delta or region != saved_region:
            enqueue(row, col)
            continue
        labels[row, col] = region
        boundary[row, col] = is_boundary
        totals[region] += image[row, col]
        counts[region] += 1
        assigned += 1
        if on_step is not None:
            on_step(labels, assigned)
        for r, c in _b_neighbors(row, col, image.shape, connectivity):
            enqueue(r, c)
    return labels, boundary


## 3. Region Growing — tự cài đặt

Các hàm được định nghĩa trong các ô code dưới đây.

### `region_growing_classic(img, seeds, T, order="stack")`

**Hợp đồng đầu ra:** Mảng nhãn int32; -1 cho pixel chưa thuộc vùng; hỗ trợ stack (LIFO) và queue (FIFO).

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; chưa chạy thí nghiệm trên ảnh tự chọn.

In [2]:
def region_growing_classic(img, seeds, T, order="stack"):
    """Phát triển từng vùng, lân cận 4; ngưỡng so với trung bình cập nhật.

    Khởi tạo tất cả seed trước, xử lý vùng theo thứ tự seeds. Pixel bị
    từ chối có thể được xét lại từ một pixel lân cận khác được nhận sau đó.
    """
    image, seeds = _b_inputs(img, seeds)
    if not np.isfinite(T) or T < 0:
        raise ValueError("T phải không âm và hữu hạn")
    if order not in ("stack", "queue"):
        raise ValueError("order phải là stack hoặc queue")
    labels = np.full(image.shape, -1, dtype=np.int32)
    for region, (row, col) in enumerate(seeds):
        labels[row, col] = region
    for region, (row, col) in enumerate(seeds):
        total, count = float(image[row, col]), 1
        frontier = deque(_b_neighbors(row, col, image.shape))
        while frontier:
            r, c = frontier.pop() if order == "stack" else frontier.popleft()
            if labels[r, c] >= 0:
                continue
            if abs(float(image[r, c]) - total / count) <= T:
                labels[r, c] = region
                total += image[r, c]
                count += 1
                frontier.extend(_b_neighbors(r, c, image.shape))
    return labels


## 4. SRG và hàm hỗ trợ — tự cài đặt

Các hàm được định nghĩa trong các ô code dưới đây.

### `srg(img, seeds, connectivity=4, stale="lazy", tie="insertion")`

**Hợp đồng đầu ra:** (labels, boundary_mask); nhãn cuối cùng đều ≥ 0; dùng heapq; các lựa chọn stale/tie theo đề.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; chưa chạy thí nghiệm trên ảnh tự chọn.

In [3]:
def srg(img, seeds, connectivity=4, stale="lazy", tie="insertion"):
    """Trả (labels int32, boundary_mask bool), mọi pixel cuối cùng có nhãn.

    lazy: tính lại delta khi pop theo trung bình vùng hiện tại, requeue nếu
    thay đổi. frozen: so với cường độ seed. Heap lazy không quét lại toàn
    bộ frontier khi trung bình đổi; ưu tiên là các khóa hiện có trong heap.
    boundary_mask ghi pixel kề nhiều vùng tại thời điểm gán, không gồm seed.
    random dùng np.random.seed để tái lập; ưu tiên ngẫu nhiên cố định trong lần chạy.
    """
    return _b_srg_engine(img, seeds, connectivity, stale, tie)


### `srg_step_frames(img, seeds, n_frames=8, **kwargs)`

**Hợp đồng đầu ra:** Danh sách n_frames ảnh nhãn tại các mốc tiến trình; các ảnh là bản chụp độc lập.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; chưa chạy thí nghiệm trên ảnh tự chọn.

In [4]:
def srg_step_frames(img, seeds, n_frames=8, **kwargs):
    """Chụp đúng n_frames bản sao nhãn theo số pixel đã gán; frame cuối hoàn tất.

    n_frames=1 chỉ chụp kết quả cuối. Nhiều mốc trùng nhau trên ảnh nhỏ
    vẫn tạo các bản sao độc lập. kwargs hỗ trợ connectivity, stale và tie.
    """
    image, seeds = _b_inputs(img, seeds)
    n_frames = _b_integer(n_frames, "n_frames", 1)
    targets = (np.array([image.size]) if n_frames == 1 else
               np.ceil(np.linspace(len(seeds), image.size, n_frames)).astype(int))
    frames = []
    def capture(labels, assigned):
        while len(frames) < n_frames and assigned >= targets[len(frames)]:
            frames.append(labels.copy())
    _b_srg_engine(image, seeds, on_step=capture, **kwargs)
    return frames


### `seed_sensitivity(img, seeds, segment_fn, n_trials=20, jitter=3)`

**Hợp đồng đầu ra:** Bản đồ tần suất đổi nhãn khi dịch seeds trong khoảng ±jitter; ghi quy ước nhãn và cách xử lý jitter không hợp lệ.

**Trạng thái:** Đã cài đặt trong ô code tiếp theo; chưa chạy thí nghiệm trên ảnh tự chọn.

In [5]:
def seed_sensitivity(img, seeds, segment_fn, n_trials=20, jitter=3):
    """Tần suất đổi vùng theo định danh seed, so với phân vùng gốc.

    segment_fn(image, seeds) trả labels hoặc (labels, boundary_mask).
    Nhãn tại các seed phải khác nhau; pixel chưa gán giữ mã -1. Dịch seed
    trong cửa sổ nguyên ±jitter, giới hạn trong ảnh, không trùng; tránh các
    vị trí seed gốc còn lại để luôn có lựa chọn dự phòng. Dùng np.random.seed
    để tái lập; randomness riêng trong segment_fn cũng tham gia phép đo.
    """
    image, seeds = _b_inputs(img, seeds)
    n_trials = _b_integer(n_trials, "n_trials", 1)
    jitter = _b_integer(jitter, "jitter")
    if not callable(segment_fn):
        raise ValueError("segment_fn phải là callable")
    def segment(points):
        output = segment_fn(image.copy(), points.copy())
        labels = np.asarray(output[0] if isinstance(output, tuple) else output)
        if (labels.shape != image.shape or not np.issubdtype(labels.dtype, np.integer)
                or np.any(labels < -1)):
            raise ValueError("segment_fn phải trả ảnh nhãn nguyên cùng shape; nhãn >= -1")
        ids = labels[points[:, 0], points[:, 1]]
        if np.any(ids < 0) or len(np.unique(ids)) != len(points):
            raise ValueError("mỗi seed phải thuộc một vùng khác nhau")
        normalized = np.full(image.shape, -1, dtype=np.int32)
        for region, label in enumerate(ids):
            normalized[labels == label] = region
        if np.any((labels >= 0) & (normalized < 0)):
            raise ValueError("vùng có nhãn phải tương ứng với một seed")
        return normalized
    baseline = segment(seeds)
    changed = np.zeros(image.shape, dtype=np.float64)
    for _ in range(n_trials):
        moved = seeds.copy()
        occupied = {tuple(point) for point in seeds}
        for index, (row, col) in enumerate(seeds):
            occupied.remove(tuple(seeds[index]))
            candidates = [(r, c)
                          for r in range(max(0, row - jitter), min(image.shape[0], row + jitter + 1))
                          for c in range(max(0, col - jitter), min(image.shape[1], col + jitter + 1))
                          if (r, c) not in occupied]
            choice = candidates[np.random.randint(len(candidates))] if candidates else (row, col)
            moved[index] = choice
            occupied.add(choice)
        changed += segment(moved) != baseline
    return changed / n_trials


## 5. Kiểm chứng phần B

Ghi biểu diễn seeds, định danh vùng, thứ tự lân cận và miền cường độ. Căn chỉnh nhãn theo cùng hạt giống trước khi đếm pixel khác nhau; không nhầm đổi mã nhãn với đổi phân vùng.

### 5.1. Phân hoạch đầy đủ

Kiểm tra (labels >= 0).all() trên kết quả SRG cuối cùng; pixel biên vẫn phải có nhãn.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Ảnh / seeds | Pixel chưa gán | Điều kiện đạt | Artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 5.2. Bảo toàn số vùng

Số nhãn phân biệt bằng số seeds hợp lệ, không trùng.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Ảnh | Số seeds hợp lệ | Số nhãn | Đạt / chưa đạt |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 5.3. Đảo thứ tự seeds

Căn chỉnh nhãn trước khi đo; ghi tác động của thứ tự chèn khi có ties.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Thứ tự seeds / tie | Cách căn chỉnh | Pixel khác nhau | Artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 5.4. Stack / queue / SRG

Cùng ảnh và seeds; so từng cặp, thể hiện pixel chưa gán của phương pháp cổ điển.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Cặp phương pháp | Tham số | Pixel khác nhau | Artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 5.5. Ảnh phẳng với hai seeds

Ghi hình dạng vùng và quy tắc phá hòa; nếu dùng random phải ghi seed ngẫu nhiên.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Quy tắc tie | Seed ngẫu nhiên | Hình dạng / sai khác | Artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 5.6. Frames và độ nhạy seeds

Lưu ảnh tiến trình độc lập; ghi n_trials, jitter, căn chỉnh nhãn, xử lý seeds trùng/ngoài ảnh và ý nghĩa tần suất.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Cấu hình | Mốc frames / số lần thử | Định nghĩa bản đồ | Artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

## 6. Thí nghiệm — SRG so với Region Growing

Giữ cùng ảnh và seeds; ghi ngưỡng T cho Region Growing, cách căn chỉnh nhãn và pixel chưa được gán.

### Phiếu thiết kế — nhóm quyết định trước khi chạy

| Trường | Nội dung do nhóm tự điền |
| --- | --- |
| Câu hỏi và giả thuyết | Chưa điền |
| Ảnh, nguồn, lý do chọn và chỉnh sửa | Chưa chọn |
| Biến độc lập và tập giá trị quét | Chưa chọn |
| Yếu tố giữ cố định | Chưa điền |
| Phép đo, định nghĩa, đơn vị và mẫu số | Chưa chọn |
| Seed ngẫu nhiên, số lần lặp, cách tổng hợp | Chưa chọn |
| Bằng chứng cần lưu: ảnh, bảng, đường cong | Chưa điền |

### Kết quả — chỉ điền số đo thật

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.


| Lần chạy | Giá trị tham số quét | Cấu hình cố định | Phương pháp | Đại lượng / đơn vị | Giá trị | Artifact |
| --- | --- | --- | --- | --- | --- | --- |
| Chưa chạy | Chưa chọn | Chưa ghi | Chưa ghi | Chưa chọn | Chưa đo | Chưa có |

Ô dưới để bổ sung khung chạy / vẽ sau khi nhóm chốt thiết kế.


### Phân tích và giới hạn — nhóm tự viết

- Bằng chứng khác biệt giữa hai phương pháp: chưa có.
- Đường cong / điểm chuyển hành vi khi quét tham số: chưa có.
- Đại lượng nào thay đổi, tại bước nào, vì sao gây ra hiện tượng: chưa phân tích.
- Kết quả có khớp giả thuyết không, phạm vi áp dụng và giới hạn: chưa kết luận.

- [ ] Có ít nhất một phép đo và bảng số gốc.
- [ ] Có đường cong quét tham số, tên trục, đơn vị và chú giải.
- [ ] Giải thích cơ chế thay vì chỉ nói phương pháp nào tốt hơn.


### 6.1. So sánh lazy / frozen

Bố trí so sánh hai chế độ stale theo định nghĩa đề; nhóm tự chọn ảnh và phép đo, ghi các yếu tố giữ cố định.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Chế độ stale | Cấu hình cố định | Đại lượng / giá trị thực | Giải thích / artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

### 6.2. Ảnh hưởng phá hòa

So insertion / raster / random theo đề; cần ít nhất một ảnh mà thay quy tắc làm đổi phân vùng, có bằng chứng định lượng và seed để tái lập.

**Ngữ cảnh 5W1H — điền khi có kết quả thực:**
- What: đại lượng, đơn vị, ý nghĩa — chưa điền.
- Why: câu hỏi cần trả lời — chưa điền.
- When: thời điểm, lần chạy, seed — chưa điền.
- Where: ảnh, thiết bị và đường dẫn artifact — chưa điền.
- Who: người thực hiện và cấu hình thuật toán — chưa điền.
- How: tham số, cách tính, số lần lặp và tổng hợp — chưa điền.

| Quy tắc tie | Seed ngẫu nhiên | Pixel khác sau căn chỉnh | Giải thích / artifact |
| --- | --- | --- | --- |
| Chưa đo | Chưa đo | Chưa đo | Chưa đo |

Ô dưới dành cho mã kiểm chứng do nhóm bổ sung. Chưa có output hoặc kết luận.

**Nhận xét kiểm chứng:** Chưa thực hiện.

- [ ] Có hình / bảng thực tế và cấu hình tái lập.
- [ ] Đối chiếu tiêu chí đề; giải thích sai khác hoặc giới hạn.

## Tổng kết và checklist phần B

**Kết luận / hạn chế / công việc còn lại:** Chưa thực hiện.

- [ ] Đủ 4 hàm B đúng chữ ký và hợp đồng đầu ra.
- [ ] Đủ năm kiểm chứng bắt buộc, ảnh tiến trình và bản đồ seed sensitivity.
- [ ] Có so sánh stack / queue / SRG, lazy / frozen và ảnh hưởng phá hòa.
- [ ] Đủ thí nghiệm SRG so với Region Growing: bằng chứng, quét tham số, giải thích.

- [ ] Điền nguồn ảnh và tài liệu tham khảo thực tế đã dùng.
- [ ] Restart & Run All với đúng kernel, lưu đủ output thực tế.
- [ ] Trước khi nộp: ghép A và B thành một notebook, chỉnh đường dẫn ảnh theo vị trí mới.

Notebook rỗng chạy được không có nghĩa thuật toán đã qua kiểm chứng.
Đề hiện ghi tên bài nộp `bt1_<Nhom>.ipynb`; xác nhận với giảng viên trước khi đổi tiền tố.
